# DefakeX Experiment 4 — Exposure-Aware Legacy Replay and Unseen-Generator Stress Test

**Purpose:** test the Experiment 3 frozen frequency+spatial embedding fusion without confusing expert-seen data with unseen generalization.

This notebook:

- never trains or tunes on the locked 898-image benchmark;
- never retrains the experts;
- labels every dataset by its historical exposure;
- reports legacy datasets as replay/compatibility results only;
- treats Stable Diffusion as a new-generator stress test only if its images have never been used before;
- uses the exact heterogeneous preprocessing: full-image FFT for frequency and MTCNN face crop for spatial;
- uses `Experiment3_frequency_plus_spatial_embeddings.joblib`, not the metadata or old-logit model.

Historical exposure registry:

| Dataset | Historical role | Allowed interpretation now |
|---|---|---|
| StyleGAN Real/Fake | Frequency expert training | Expert-seen replay |
| Flickr30K | Frequency expert real training | Expert-seen replay |
| DeepDetect 2025 | Frequency expert training | Expert-seen replay |
| OpenFake | Frequency expert training | Expert-seen replay |
| FF++ | Spatial/fusion development | Legacy replay only |
| Legacy phone images | Spatial/fusion development | Legacy replay only |
| Celeb-DF | Previous cross-dataset evaluation | Previously touched stress test |
| Stable Diffusion faces | Not previously used, if newly attached | New-generator stress test |

In [1]:
# CELL 1 — IMPORTS, REPRODUCIBILITY, AND CONFIGURATION
import os, re, json, math, random, hashlib, shutil, zipfile, warnings, sys, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
    roc_auc_score, confusion_matrix, precision_recall_fscore_support, brier_score_loss)
import joblib

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
INPUT_ROOT=Path('/kaggle/input')
WORK_ROOT=Path('/kaggle/working')
OUTPUT_DIR=WORK_ROOT/'defakex_experiment_4'
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)

# Keep the first run computationally bounded and source-balanced.
MAX_PER_CLASS_PER_DATASET=1000
BATCH_SIZE=32
NUM_WORKERS=2
THRESHOLD=0.665  # selected from grouped DefakeX-training OOF in Experiment 3
VALID_EXT={'.jpg','.jpeg','.png','.webp','.bmp','.tif','.tiff','.jfif'}

# Set False only if Stable Diffusion images were used anywhere in earlier expert/fusion training.
STABLE_DIFFUSION_CONFIRMED_NEVER_USED=True

print('Device:',DEVICE)
print('Output:',OUTPUT_DIR)

Device: cuda
Output: /kaggle/working/defakex_experiment_4


In [2]:
# CELL 2 — SAFE FACENET-PYTORCH INSTALLATION

import sys
import subprocess

try:
    from facenet_pytorch import MTCNN
    print("facenet-pytorch is already available.")

except ModuleNotFoundError:
    print("Installing facenet-pytorch without modifying Kaggle dependencies...")

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-deps",
        "--only-binary=:all:",
        "facenet-pytorch==2.6.0",
    ])

    from facenet_pytorch import MTCNN

print("PASS: facenet-pytorch is available.")
print("Torch:", torch.__version__)
print("Torchvision:", __import__("torchvision").__version__)

Installing facenet-pytorch without modifying Kaggle dependencies...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 41.3 MB/s eta 0:00:00
PASS: facenet-pytorch is available.
Torch: 2.11.0+cu128
Torchvision: 0.26.0+cu128


In [3]:
# CELL 3 — USE EXACT CHECKPOINT AND EXPERIMENT 3 MODEL PATHS

from pathlib import Path

MODEL_ROOT = Path(
    "/kaggle/input/datasets/kashirhanif/frequency-model-checkpoint"
)

EXPERIMENT3_ROOT = Path(
    "/kaggle/input/datasets/kashirhanif/experiment-result/"
    "DefakeX_Experiment3_Embedding_Fusion_Hardening"
)

FREQUENCY_CKPT = MODEL_ROOT / "best_model.pth"
SPATIAL_CKPT = MODEL_ROOT / "best_model_spatial_v2.pth"

FUSION_MODEL = (
    EXPERIMENT3_ROOT
    / "Experiment3_frequency_plus_spatial_embeddings.joblib"
)

assert FREQUENCY_CKPT.exists(), (
    f"Frequency checkpoint not found:\n{FREQUENCY_CKPT}"
)

assert SPATIAL_CKPT.exists(), (
    f"Spatial checkpoint not found:\n{SPATIAL_CKPT}"
)

assert FUSION_MODEL.exists(), (
    f"Experiment 3 fusion model not found:\n{FUSION_MODEL}"
)

print("Resolved required models:")
print("Frequency:", FREQUENCY_CKPT)
print("Spatial:  ", SPATIAL_CKPT)
print("Fusion:   ", FUSION_MODEL)

Resolved required models:
Frequency: /kaggle/input/datasets/kashirhanif/frequency-model-checkpoint/best_model.pth
Spatial:   /kaggle/input/datasets/kashirhanif/frequency-model-checkpoint/best_model_spatial_v2.pth
Fusion:    /kaggle/input/datasets/kashirhanif/experiment-result/DefakeX_Experiment3_Embedding_Fusion_Hardening/Experiment3_frequency_plus_spatial_embeddings.joblib


In [5]:
# CELL 4 — EXACT DATASET PATHS AND EXPOSURE REGISTRY

from pathlib import Path
import pandas as pd


# ---------------------------------------------------------
# Exact paths verified in Kaggle
# ---------------------------------------------------------

STYLEGAN_ROOT = Path(
    "/kaggle/input/datasets/xhlulu/"
    "140k-real-and-fake-faces/"
    "real_vs_fake/real-vs-fake/train"
)

FLICKR30K_ROOT = Path(
    "/kaggle/input/datasets/adityajn105/"
    "flickr30k/Images/flickr30k_images"
)

DEEPDETECT_ROOT = Path(
    "/kaggle/input/datasets/ayushmandatta1/"
    "deepdetect-2025/ddata/train"
)

FFPP_ROOT = Path(
    "/kaggle/input/datasets/gradientvoyager/"
    "faceforensics-c23-extracted-faces-100k/"
    "dataset_processed_split"
)

PHONE_ROOT = Path(
    "/kaggle/input/datasets/kashirhanif/phone-images"
)

CELEBDF_ROOT = Path(
    "/kaggle/input/datasets/pranabr0y/"
    "celebdf-v2image-dataset/Celeb_V2"
)

STABLE_DIFFUSION_ROOT = Path(
    "/kaggle/input/datasets/mohannadaymansalah/"
    "stable-diffusion-dataaaaaaaaa"
)

# OpenFake consists of four separate Kaggle datasets.
OPENFAKE_ROOTS = [
    Path(
        f"/kaggle/input/datasets/kashirhanif/"
        f"openfake-part-{part}"
    )
    for part in range(1, 5)
]


# ---------------------------------------------------------
# Dataset exposure registry
# ---------------------------------------------------------

DATASETS = [
    {
        "dataset": "stylegan",
        "roots": [STYLEGAN_ROOT],
        "exposure": "frequency_expert_seen",
        "role": "legacy_replay",
    },
    {
        "dataset": "flickr30k",
        "roots": [FLICKR30K_ROOT],
        "exposure": "frequency_expert_seen",
        "role": "legacy_replay",
    },
    {
        "dataset": "deepdetect2025",
        "roots": [DEEPDETECT_ROOT],
        "exposure": "frequency_expert_seen",
        "role": "legacy_replay",
    },
    {
        "dataset": "openfake",
        "roots": OPENFAKE_ROOTS,
        "exposure": "frequency_expert_seen",
        "role": "legacy_replay",
    },
    {
        "dataset": "ffpp",
        "roots": [FFPP_ROOT],
        "exposure": "spatial_or_fusion_seen",
        "role": "legacy_replay",
    },
    {
        "dataset": "legacy_phone",
        "roots": [PHONE_ROOT],
        "exposure": "fusion_seen",
        "role": "legacy_replay",
    },
    {
        "dataset": "celebdf",
        "roots": [CELEBDF_ROOT],
        "exposure": "previously_evaluated",
        "role": "touched_stress_test",
    },
    {
        "dataset": "stable_diffusion",
        "roots": [STABLE_DIFFUSION_ROOT],
        "exposure": (
            "never_used"
            if STABLE_DIFFUSION_CONFIRMED_NEVER_USED
            else "historically_used"
        ),
        "role": (
            "new_generator_stress"
            if STABLE_DIFFUSION_CONFIRMED_NEVER_USED
            else "legacy_replay"
        ),
    },
]


# ---------------------------------------------------------
# Validate paths and construct discovery table
# ---------------------------------------------------------

discovery_rows = []

for item in DATASETS:
    existing_roots = [
        root
        for root in item["roots"]
        if root is not None and root.exists()
    ]

    missing_roots = [
        str(root)
        for root in item["roots"]
        if root is not None and not root.exists()
    ]

    discovery_rows.append({
        "dataset": item["dataset"],
        "aliases": "",
        "exposure": item["exposure"],
        "role": item["role"],
        "found": len(existing_roots) > 0,
        "selected_root": (
            str(existing_roots[0])
            if existing_roots
            else ""
        ),
        "selected_roots": [
            str(root)
            for root in existing_roots
        ],
        "missing_roots": " | ".join(missing_roots),
        "other_candidates": "",
    })


discovery = pd.DataFrame(discovery_rows)

display(
    discovery[
        [
            "dataset",
            "exposure",
            "role",
            "found",
            "selected_roots",
            "missing_roots",
        ]
    ]
)

discovery.to_csv(
    OUTPUT_DIR / "Experiment4_Dataset_Discovery.csv",
    index=False,
)


# ---------------------------------------------------------
# Final verification
# ---------------------------------------------------------

print("\nDataset path verification:")

for row in discovery.to_dict("records"):
    status = "FOUND" if row["found"] else "NOT FOUND"

    print(f"\n{row['dataset']}: {status}")

    for root in row["selected_roots"]:
        print("  ✓", root)

    if row["missing_roots"]:
        print("  ✗", row["missing_roots"])


assert discovery["found"].all(), (
    "One or more dataset paths were not found. "
    "Review the dataset verification output above."
)

assert len(
    discovery.loc[
        discovery["dataset"] == "openfake",
        "selected_roots",
    ].iloc[0]
) == 4, "All four OpenFake dataset parts were not found."


print("\nPASS: All required datasets were found.")
print("PASS: All four OpenFake parts were found.")
print("PASS: Stable Diffusion dataset was found.")

,dataset,exposure,role,found,selected_roots,missing_roots
0,stylegan,frequency_expert_seen,legacy_replay,True,[/kaggle/input/datasets/xhlulu/140k-real-and-f...,
1,flickr30k,frequency_expert_seen,legacy_replay,True,[/kaggle/input/datasets/adityajn105/flickr30k/...,
2,deepdetect2025,frequency_expert_seen,legacy_replay,True,[/kaggle/input/datasets/ayushmandatta1/deepdet...,
3,openfake,frequency_expert_seen,legacy_replay,True,[/kaggle/input/datasets/kashirhanif/openfake-p...,
4,ffpp,spatial_or_fusion_seen,legacy_replay,True,[/kaggle/input/datasets/gradientvoyager/facefo...,
5,legacy_phone,fusion_seen,legacy_replay,True,[/kaggle/input/datasets/kashirhanif/phone-images],
6,celebdf,previously_evaluated,touched_stress_test,True,[/kaggle/input/datasets/pranabr0y/celebdf-v2im...,
7,stable_diffusion,never_used,new_generator_stress,True,[/kaggle/input/datasets/mohannadaymansalah/sta...,



Dataset path verification:

stylegan: FOUND
  ✓ /kaggle/input/datasets/xhlulu/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/train

flickr30k: FOUND
  ✓ /kaggle/input/datasets/adityajn105/flickr30k/Images/flickr30k_images

deepdetect2025: FOUND
  ✓ /kaggle/input/datasets/ayushmandatta1/deepdetect-2025/ddata/train

openfake: FOUND
  ✓ /kaggle/input/datasets/kashirhanif/openfake-part-1
  ✓ /kaggle/input/datasets/kashirhanif/openfake-part-2
  ✓ /kaggle/input/datasets/kashirhanif/openfake-part-3
  ✓ /kaggle/input/datasets/kashirhanif/openfake-part-4

ffpp: FOUND
  ✓ /kaggle/input/datasets/gradientvoyager/faceforensics-c23-extracted-faces-100k/dataset_processed_split

legacy_phone: FOUND
  ✓ /kaggle/input/datasets/kashirhanif/phone-images

celebdf: FOUND
  ✓ /kaggle/input/datasets/pranabr0y/celebdf-v2image-dataset/Celeb_V2

stable_diffusion: FOUND
  ✓ /kaggle/input/datasets/mohannadaymansalah/stable-diffusion-dataaaaaaaaa

PASS: All required datasets were found.
PASS: All four OpenFake

### Label-resolution rule

The next cell supports ordinary `real/`, `fake/`, `0_real/`, `1_fake/`, `original/`, `manipulated/`, generator, and metadata-based layouts. Ambiguous files are **not guessed**: they are written to a CSV and skipped. This is safer than silently assigning incorrect labels.

In [6]:
# CELL 4B — EXTRACT A BALANCED OPENFAKE REPLAY SAMPLE FROM PARQUET

import io
import gc
import heapq
import hashlib
from collections import defaultdict
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
from PIL import Image


OPENFAKE_EXTRACT_ROOT = Path(
    "/kaggle/working/experiment4_openfake_images"
)

OPENFAKE_EXTRACT_MANIFEST = Path(
    "/kaggle/working/experiment4_openfake_manifest.csv"
)

# Four parts × 250 per class = 1,000 real + 1,000 fake.
OPENFAKE_PER_PART_PER_CLASS = 250


def openfake_priority(
    part_number,
    parquet_path,
    row_index,
):
    value = (
        f"{SEED}|openfake|"
        f"{part_number}|"
        f"{parquet_path}|"
        f"{row_index}"
    )

    digest = hashlib.blake2b(
        value.encode("utf-8"),
        digest_size=8,
    ).digest()

    return int.from_bytes(
        digest,
        byteorder="big",
        signed=False,
    )


def add_openfake_candidate(
    heaps,
    key,
    candidate,
    limit,
):
    priority = candidate["priority"]

    heap_item = (
        -priority,
        str(candidate["parquet_path"]),
        candidate["row_index"],
        candidate,
    )

    heap = heaps.setdefault(key, [])

    if len(heap) < limit:
        heapq.heappush(
            heap,
            heap_item,
        )

    else:
        worst_priority = -heap[0][0]

        if priority < worst_priority:
            heapq.heapreplace(
                heap,
                heap_item,
            )


def openfake_cache_is_valid():
    if not OPENFAKE_EXTRACT_MANIFEST.exists():
        return False

    try:
        cached = pd.read_csv(
            OPENFAKE_EXTRACT_MANIFEST
        )

        if len(cached) == 0:
            return False

        required_columns = {
            "path",
            "label",
            "source",
            "part",
        }

        if not required_columns.issubset(
            cached.columns
        ):
            return False

        existing = cached["path"].map(
            lambda path: Path(path).exists()
        )

        if not existing.all():
            return False

        real_count = (
            cached["label"].astype(int) == 0
        ).sum()

        fake_count = (
            cached["label"].astype(int) == 1
        ).sum()

        return (
            real_count >= 1_000
            and fake_count >= 1_000
        )

    except Exception:
        return False


def extract_openfake_replay_sample(
    force=False,
):
    if (
        not force
        and openfake_cache_is_valid()
    ):
        cached = pd.read_csv(
            OPENFAKE_EXTRACT_MANIFEST
        )

        print(
            "Using existing OpenFake extraction:",
            OPENFAKE_EXTRACT_MANIFEST,
            flush=True,
        )

        print(
            pd.crosstab(
                cached["part"],
                cached["label"],
            ),
            flush=True,
        )

        return cached

    print(
        "Building deterministic OpenFake sample...",
        flush=True,
    )

    parquet_records = []

    for part_number, root in enumerate(
        OPENFAKE_ROOTS,
        start=1,
    ):
        parquet_files = sorted(
            root.rglob("*.parquet")
        )

        print(
            f"OpenFake part {part_number}: "
            f"{len(parquet_files)} Parquet files",
            flush=True,
        )

        if not parquet_files:
            raise FileNotFoundError(
                f"No Parquet files found in "
                f"OpenFake part {part_number}: "
                f"{root}"
            )

        for parquet_path in parquet_files:
            parquet_records.append({
                "part": part_number,
                "path": parquet_path,
            })

    # -----------------------------------------------------
    # Pass 1: select 250 real and 250 fake per part
    # without decoding all images.
    # -----------------------------------------------------

    selection_heaps = {}

    for file_index, record in enumerate(
        parquet_records,
        start=1,
    ):
        part_number = record["part"]
        parquet_path = record["path"]

        print(
            f"[Metadata {file_index}/"
            f"{len(parquet_records)}] "
            f"Part {part_number}: "
            f"{parquet_path.name}",
            flush=True,
        )

        try:
            metadata = pq.read_table(
                parquet_path,
                columns=[
                    "label",
                    "model",
                ],
            )

            labels = (
                metadata["label"]
                .to_pylist()
            )

            models = (
                metadata["model"]
                .to_pylist()
            )

        except Exception as error:
            print(
                "  Metadata read failed:",
                type(error).__name__,
                str(error),
                flush=True,
            )

            continue

        for row_index, (
            raw_label,
            raw_model,
        ) in enumerate(
            zip(labels, models)
        ):
            label_text = (
                str(raw_label)
                .strip()
                .lower()
            )

            is_real = label_text == "real"
            label = 0 if is_real else 1

            source = (
                "real"
                if is_real
                else str(raw_model).strip()
            )

            candidate = {
                "part": part_number,
                "parquet_path": parquet_path,
                "row_index": row_index,
                "label": label,
                "source": source,
                "priority": openfake_priority(
                    part_number,
                    parquet_path,
                    row_index,
                ),
            }

            add_openfake_candidate(
                heaps=selection_heaps,
                key=(
                    part_number,
                    label,
                ),
                candidate=candidate,
                limit=(
                    OPENFAKE_PER_PART_PER_CLASS
                ),
            )

        del metadata
        del labels
        del models
        gc.collect()

        selected_real = sum(
            len(
                selection_heaps.get(
                    (part, 0),
                    [],
                )
            )
            for part in range(1, 5)
        )

        selected_fake = sum(
            len(
                selection_heaps.get(
                    (part, 1),
                    [],
                )
            )
            for part in range(1, 5)
        )

        print(
            f"  Selected so far: "
            f"real={selected_real:,}, "
            f"fake={selected_fake:,}",
            flush=True,
        )

    selected_candidates = []

    for heap in selection_heaps.values():
        selected_candidates.extend(
            item[3]
            for item in heap
        )

    selection_table = pd.DataFrame(
        selected_candidates
    )

    assert len(selection_table), (
        "No OpenFake records were selected."
    )

    print(
        "\nOpenFake metadata selection:",
        flush=True,
    )

    print(
        pd.crosstab(
            selection_table["part"],
            selection_table["label"],
            margins=True,
        ),
        flush=True,
    )

    # Verify that each part contributed both classes.
    for part_number in range(1, 5):
        for label in (0, 1):
            count = len(
                selection_table[
                    (
                        selection_table["part"]
                        == part_number
                    )
                    & (
                        selection_table["label"]
                        == label
                    )
                ]
            )

            assert count == (
                OPENFAKE_PER_PART_PER_CLASS
            ), (
                f"OpenFake part {part_number}, "
                f"label {label}: selected "
                f"{count}, expected "
                f"{OPENFAKE_PER_PART_PER_CLASS}."
            )

    # -----------------------------------------------------
    # Pass 2: read and save only selected image bytes.
    # -----------------------------------------------------

    OPENFAKE_EXTRACT_ROOT.mkdir(
        parents=True,
        exist_ok=True,
    )

    grouped_candidates = defaultdict(list)

    for candidate in selected_candidates:
        grouped_candidates[
            str(candidate["parquet_path"])
        ].append(candidate)

    output_rows = []
    saved_count = 0
    failed_rows = []

    grouped_items = sorted(
        grouped_candidates.items(),
        key=lambda item: item[0],
    )

    for file_index, (
        parquet_path_string,
        candidates,
    ) in enumerate(
        grouped_items,
        start=1,
    ):
        parquet_path = Path(
            parquet_path_string
        )

        print(
            f"\n[Images {file_index}/"
            f"{len(grouped_items)}] "
            f"Reading {parquet_path.name}",
            flush=True,
        )

        print(
            f"  Selected rows in this file: "
            f"{len(candidates):,}",
            flush=True,
        )

        try:
            image_table = pq.read_table(
                parquet_path,
                columns=["image"],
            )

            images = image_table["image"]

        except Exception as error:
            print(
                "  Image-column read failed:",
                type(error).__name__,
                str(error),
                flush=True,
            )

            for candidate in candidates:
                failed_rows.append({
                    **candidate,
                    "error": (
                        f"{type(error).__name__}: "
                        f"{error}"
                    ),
                })

            continue

        for local_index, candidate in enumerate(
            sorted(
                candidates,
                key=lambda row: (
                    row["part"],
                    row["label"],
                    row["priority"],
                ),
            ),
            start=1,
        ):
            try:
                raw_image = images[
                    candidate["row_index"]
                ].as_py()

                if isinstance(raw_image, dict):
                    raw_image = raw_image.get(
                        "bytes"
                    )

                if raw_image is None:
                    raise ValueError(
                        "Image bytes are missing."
                    )

                label_name = (
                    "real"
                    if candidate["label"] == 0
                    else "fake"
                )

                destination_directory = (
                    OPENFAKE_EXTRACT_ROOT
                    / label_name
                    / f"part_{candidate['part']}"
                )

                destination_directory.mkdir(
                    parents=True,
                    exist_ok=True,
                )

                filename = (
                    f"of_p{candidate['part']}_"
                    f"{candidate['priority']:020d}.jpg"
                )

                destination_path = (
                    destination_directory
                    / filename
                )

                with Image.open(
                    io.BytesIO(raw_image)
                ) as image:
                    image = image.convert("RGB")

                    image.save(
                        destination_path,
                        format="JPEG",
                        quality=88,
                    )

                output_rows.append({
                    "path": str(
                        destination_path
                    ),
                    "label": int(
                        candidate["label"]
                    ),
                    "source": candidate[
                        "source"
                    ],
                    "part": int(
                        candidate["part"]
                    ),
                    "original_parquet": str(
                        parquet_path
                    ),
                    "original_row": int(
                        candidate["row_index"]
                    ),
                })

                saved_count += 1

                if saved_count % 100 == 0:
                    print(
                        f"  Saved "
                        f"{saved_count:,}/"
                        f"{len(selected_candidates):,}",
                        flush=True,
                    )

            except Exception as error:
                failed_rows.append({
                    "part": candidate["part"],
                    "parquet_path": str(
                        parquet_path
                    ),
                    "row_index": candidate[
                        "row_index"
                    ],
                    "label": candidate[
                        "label"
                    ],
                    "source": candidate[
                        "source"
                    ],
                    "error": (
                        f"{type(error).__name__}: "
                        f"{error}"
                    ),
                })

        del images
        del image_table
        gc.collect()

    openfake_manifest = pd.DataFrame(
        output_rows
    )

    openfake_manifest.to_csv(
        OPENFAKE_EXTRACT_MANIFEST,
        index=False,
    )

    if failed_rows:
        pd.DataFrame(
            failed_rows
        ).to_csv(
            "/kaggle/working/"
            "experiment4_openfake_failures.csv",
            index=False,
        )

    print(
        "\nOpenFake extraction completed:",
        len(openfake_manifest),
        flush=True,
    )

    print(
        pd.crosstab(
            openfake_manifest["part"],
            openfake_manifest["label"],
            margins=True,
        ),
        flush=True,
    )

    real_count = (
        openfake_manifest["label"] == 0
    ).sum()

    fake_count = (
        openfake_manifest["label"] == 1
    ).sum()

    assert real_count == 1_000, (
        f"Expected 1,000 OpenFake real images, "
        f"but extracted {real_count}."
    )

    assert fake_count == 1_000, (
        f"Expected 1,000 OpenFake fake images, "
        f"but extracted {fake_count}."
    )

    return openfake_manifest


openfake_extracted = (
    extract_openfake_replay_sample(
        force=False
    )
)


# Update the live discovery table so Cell 5 scans the
# extracted JPEG folder instead of the Parquet roots.
openfake_index = discovery.index[
    discovery["dataset"] == "openfake"
].tolist()

assert len(openfake_index) == 1, (
    "Could not uniquely identify the OpenFake "
    "row in the discovery dataframe."
)

openfake_index = openfake_index[0]

discovery.at[
    openfake_index,
    "selected_root",
] = str(OPENFAKE_EXTRACT_ROOT)

discovery.at[
    openfake_index,
    "selected_roots",
] = [str(OPENFAKE_EXTRACT_ROOT)]

discovery.at[
    openfake_index,
    "found",
] = True


print(
    "\nPASS: Cell 5 will now use:",
    discovery.at[
        openfake_index,
        "selected_roots",
    ],
    flush=True,
)

Building deterministic OpenFake sample...
OpenFake part 1: 24 Parquet files
OpenFake part 2: 19 Parquet files
OpenFake part 3: 24 Parquet files
OpenFake part 4: 22 Parquet files
[Metadata 1/89] Part 1: test-00000-of-00007.parquet
  Selected so far: real=250, fake=250
[Metadata 2/89] Part 1: test-00001-of-00007.parquet
  Selected so far: real=250, fake=250
[Metadata 3/89] Part 1: test-00002-of-00007.parquet
  Selected so far: real=250, fake=250
[Metadata 4/89] Part 1: test-00003-of-00007.parquet
  Selected so far: real=250, fake=250
[Metadata 5/89] Part 1: test-00004-of-00007.parquet
  Selected so far: real=250, fake=250
[Metadata 6/89] Part 1: test-00005-of-00007.parquet
  Selected so far: real=250, fake=250
[Metadata 7/89] Part 1: test-00006-of-00007.parquet
  Selected so far: real=250, fake=250
[Metadata 8/89] Part 1: train-00000-of-00206.parquet
  Selected so far: real=250, fake=250
[Metadata 9/89] Part 1: train-00001-of-00206.parquet
  Selected so far: real=250, fake=250
[Metadata 

/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1136: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(



[Images 80/89] Reading train-00072-of-00206.parquet
  Selected rows in this file: 16
  Saved 1,800/2,000

[Images 81/89] Reading train-00073-of-00206.parquet
  Selected rows in this file: 25

[Images 82/89] Reading train-00074-of-00206.parquet
  Selected rows in this file: 15

[Images 83/89] Reading train-00075-of-00206.parquet
  Selected rows in this file: 19

[Images 84/89] Reading train-00076-of-00206.parquet
  Selected rows in this file: 18

[Images 85/89] Reading train-00077-of-00206.parquet
  Selected rows in this file: 24
  Saved 1,900/2,000

[Images 86/89] Reading train-00078-of-00206.parquet
  Selected rows in this file: 29

[Images 87/89] Reading train-00079-of-00206.parquet
  Selected rows in this file: 28

[Images 88/89] Reading train-00080-of-00206.parquet
  Selected rows in this file: 19

[Images 89/89] Reading train-00081-of-00206.parquet
  Selected rows in this file: 19

OpenFake extraction completed: 1998
label    0     1   All
part                  
1      250   250 

AssertionError: Expected 1,000 OpenFake real images, but extracted 998.

In [7]:
# CELL 4C — ACCEPT VALID OPENFAKE EXTRACTION AND UPDATE DISCOVERY

from pathlib import Path
import pandas as pd


OPENFAKE_EXTRACT_ROOT = Path(
    "/kaggle/working/experiment4_openfake_images"
)

OPENFAKE_EXTRACT_MANIFEST = Path(
    "/kaggle/working/experiment4_openfake_manifest.csv"
)


assert OPENFAKE_EXTRACT_MANIFEST.exists(), (
    "OpenFake extraction manifest was not created."
)

assert OPENFAKE_EXTRACT_ROOT.exists(), (
    "OpenFake extracted image directory was not created."
)


openfake_extracted = pd.read_csv(
    OPENFAKE_EXTRACT_MANIFEST
)

openfake_extracted["file_exists"] = (
    openfake_extracted["path"]
    .map(lambda path: Path(path).exists())
)

missing_files = openfake_extracted[
    ~openfake_extracted["file_exists"]
].copy()

valid_openfake = openfake_extracted[
    openfake_extracted["file_exists"]
].copy()


real_count = int(
    (
        valid_openfake["label"]
        .astype(int)
        == 0
    ).sum()
)

fake_count = int(
    (
        valid_openfake["label"]
        .astype(int)
        == 1
    ).sum()
)


print("Valid OpenFake extraction:")
print(" Real:", real_count)
print(" Fake:", fake_count)
print(" Total:", len(valid_openfake))
print(" Missing saved files:", len(missing_files))


# Allow a very small image-decoding failure tolerance.
assert real_count >= 990, (
    f"Too few OpenFake real images: {real_count}"
)

assert fake_count >= 990, (
    f"Too few OpenFake fake images: {fake_count}"
)


# Save a clean manifest containing only existing files.
valid_openfake.drop(
    columns=["file_exists"],
    errors="ignore",
).to_csv(
    OPENFAKE_EXTRACT_MANIFEST,
    index=False,
)


# Update the live discovery dataframe.
openfake_indexes = discovery.index[
    discovery["dataset"] == "openfake"
].tolist()

assert len(openfake_indexes) == 1, (
    "Could not uniquely locate the OpenFake "
    "entry in the discovery dataframe."
)

openfake_index = openfake_indexes[0]

discovery.at[
    openfake_index,
    "selected_root",
] = str(OPENFAKE_EXTRACT_ROOT)

discovery.at[
    openfake_index,
    "selected_roots",
] = [str(OPENFAKE_EXTRACT_ROOT)]

discovery.at[
    openfake_index,
    "found",
] = True

discovery.at[
    openfake_index,
    "missing_roots",
] = ""


print(
    "\nUpdated OpenFake discovery root:",
    discovery.at[
        openfake_index,
        "selected_roots",
    ],
)

print(
    "\nOpenFake class distribution:"
)

display(
    pd.crosstab(
        valid_openfake["part"],
        valid_openfake["label"],
        margins=True,
    )
)

print(
    "\nPASS: OpenFake extraction accepted "
    "with minor decoding tolerance."
)

Valid OpenFake extraction:
 Real: 998
 Fake: 1000
 Total: 1998
 Missing saved files: 0

Updated OpenFake discovery root: ['/kaggle/working/experiment4_openfake_images']

OpenFake class distribution:


label,0,1,All
part,,,
1,250,250,500
2,248,250,498
3,250,250,500
4,250,250,500
All,998,1000,1998



PASS: OpenFake extraction accepted with minor decoding tolerance.


In [8]:
# CELL 5 — FAST, DETERMINISTIC, MULTI-ROOT EXTERNAL MANIFEST

import os
import time
import heapq
import hashlib
from pathlib import Path

import pandas as pd


# =========================================================
# Configuration
# =========================================================

LOG_EVERY_FILES = 5_000
LOG_EVERY_SECONDS = 15
COUNT_LOG_EVERY_SECONDS = 15
MAX_AMBIGUOUS_EXAMPLES = 500


REAL_TOKENS = {
    "real",
    "original",
    "originals",
    "authentic",
    "genuine",
    "human",
    "humans",
    "flickr30k",
    "flickr",
    "0_real",
    "real_images",
}

FAKE_TOKENS = {
    "fake",
    "fakes",
    "generated",
    "synthetic",
    "manipulated",
    "deepfake",
    "deepfakes",
    "ai",
    "ai_generated",
    "stable_diffusion",
    "stable-diffusion",
    "1_fake",
    "face2face",
    "faceswap",
    "faceshifter",
    "neuraltextures",
    "celeb-synthesis",
}


# =========================================================
# General helpers
# =========================================================

def log(message=""):
    """Print logs immediately inside Kaggle."""
    print(message, flush=True)


def format_duration(seconds):
    seconds = max(0, int(seconds))

    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)

    if hours:
        return f"{hours:02d}:{minutes:02d}:{seconds:02d}"

    return f"{minutes:02d}:{seconds:02d}"


def parse_roots(record):
    """Return all roots from a discovery record."""
    roots = record.get("selected_roots", [])

    if not roots and record.get("selected_root"):
        roots = [record["selected_root"]]

    if isinstance(roots, str):
        roots = [roots]

    return [
        Path(root)
        for root in roots
        if root and Path(root).exists()
    ]


# =========================================================
# Dataset label resolver
# =========================================================

def path_label(path, dataset):
    """
    Determine the image label from its dataset and directory structure.

    Returns:
        0    -> real
        1    -> fake
        None -> ambiguous
    """
    path = Path(path)

    tokens = [
        part.strip().lower()
        for part in path.parts
    ]

    joined = "/".join(tokens)

    # Known single-class datasets
    if dataset in {"flickr30k", "legacy_phone"}:
        return 0

    if dataset == "stable_diffusion":
        return 1

   
    # Celeb-DF V2
    if dataset == "celebdf":
        if (
            "celeb-synthesis" in joined
            or "/fake/" in joined
        ):
            return 1
    
        if (
            "celeb-real" in joined
            or "youtube-real" in joined
            or "/real/" in joined
        ):
            return 0
    
        return None

    # FaceForensics++
    if dataset == "ffpp":
        fake_indicators = (
            "deepfakes",
            "face2face",
            "faceswap",
            "faceshifter",
            "neuraltextures",
            "manipulated_sequences",
            "manipulated-sequences",
            "/manipulated/",
            "/fake/",
        )

        if any(
            indicator in joined
            for indicator in fake_indicators
        ):
            return 1

        real_indicators = (
            "original_sequences",
            "original-sequences",
            "/original/",
            "/pristine/",
            "/real/",
        )

        if any(
            indicator in joined
            for indicator in real_indicators
        ):
            return 0

        return None

    # For StyleGAN, DeepDetect and OpenFake, inspect the
    # nearest directory first. This avoids misreading names
    # such as "real-vs-fake" in a higher-level directory.
    for token in reversed(tokens):
        if token in REAL_TOKENS:
            return 0

        if token in FAKE_TOKENS:
            return 1

        if (
            token.startswith("real_")
            or token.startswith("real-")
        ):
            return 0

        if (
            token.startswith("fake_")
            or token.startswith("fake-")
        ):
            return 1

    return None


# =========================================================
# Deterministic reservoir sampling
# =========================================================

def deterministic_priority(dataset, filepath):
    value = f"{SEED}|{dataset}|{filepath}"

    digest = hashlib.blake2b(
        value.encode("utf-8"),
        digest_size=8,
    ).digest()

    return int.from_bytes(
        digest,
        byteorder="big",
        signed=False,
    )


def add_to_reservoir(
    reservoir,
    key,
    row,
    limit,
):
    """
    Retain the rows with the smallest deterministic hashes.

    This produces the same sample whenever SEED and paths
    remain unchanged.
    """
    priority = deterministic_priority(
        row["dataset"],
        row["filepath"],
    )

    heap_item = (
        -priority,
        row["filepath"],
        row,
    )

    heap = reservoir.setdefault(key, [])

    if len(heap) < limit:
        heapq.heappush(
            heap,
            heap_item,
        )

        return

    current_worst_priority = -heap[0][0]

    if priority < current_worst_priority:
        heapq.heapreplace(
            heap,
            heap_item,
        )


# =========================================================
# Build scan plan
# =========================================================

scan_plan = []

for record in discovery.to_dict("records"):
    if not record["found"]:
        continue

    roots = parse_roots(record)

    if roots:
        scan_plan.append({
            "record": record,
            "roots": roots,
        })


assert scan_plan, (
    "No valid dataset roots were found. "
    "Run and verify Cell 4 first."
)

log("=" * 80)
log("EXPERIMENT 4 DATASET SCAN PLAN")
log("=" * 80)

for index, plan in enumerate(
    scan_plan,
    start=1,
):
    dataset = plan["record"]["dataset"]

    log(
        f"{index}. {dataset}: "
        f"{len(plan['roots'])} root(s)"
    )

    for root_index, root in enumerate(
        plan["roots"],
        start=1,
    ):
        log(
            f"   [{root_index}] {root}"
        )


# =========================================================
# Count images so exact percentages can be displayed
# =========================================================

def count_image_files(root):
    """
    Count supported image files without opening them.

    Periodic logs are shown even during the counting stage.
    """
    image_count = 0
    directory_count = 0

    start_time = time.time()
    last_log_time = start_time

    for _, _, filenames in os.walk(root):
        directory_count += 1

        for filename in filenames:
            if (
                Path(filename).suffix.lower()
                in VALID_EXT
            ):
                image_count += 1

        current_time = time.time()

        if (
            current_time - last_log_time
            >= COUNT_LOG_EVERY_SECONDS
        ):
            log(
                "    Counting progress: "
                f"{directory_count:,} directories, "
                f"{image_count:,} images found, "
                f"elapsed "
                f"{format_duration(current_time - start_time)}"
            )

            last_log_time = current_time

    return image_count


log("\n" + "=" * 80)
log("STAGE 1 — COUNTING IMAGE FILES")
log("=" * 80)

count_start_time = time.time()

grand_total_expected = 0
dataset_expected_counts = {}
root_expected_counts = {}

for dataset_index, plan in enumerate(
    scan_plan,
    start=1,
):
    record = plan["record"]
    dataset = record["dataset"]
    roots = plan["roots"]

    dataset_total = 0

    log(
        f"\n[{dataset_index}/{len(scan_plan)}] "
        f"Counting {dataset}..."
    )

    for root_index, root in enumerate(
        roots,
        start=1,
    ):
        root_start_time = time.time()

        log(
            f"  Part/root "
            f"{root_index}/{len(roots)}"
        )
        log(f"  Path: {root}")

        root_count = count_image_files(root)

        root_expected_counts[
            (dataset, str(root))
        ] = root_count

        dataset_total += root_count
        grand_total_expected += root_count

        log(
            f"  Found {root_count:,} images "
            f"in {format_duration(time.time() - root_start_time)}"
        )

    dataset_expected_counts[
        dataset
    ] = dataset_total

    log(
        f"  Dataset total: "
        f"{dataset_total:,} images"
    )


log("\n" + "=" * 80)
log(
    f"TOTAL IMAGES TO SCAN: "
    f"{grand_total_expected:,}"
)
log(
    "Counting completed in "
    f"{format_duration(time.time() - count_start_time)}"
)
log("=" * 80)


# =========================================================
# Scan and sample images
# =========================================================

reservoir = {}
ambiguous_examples = []
scan_summary_rows = []

total_scanned = 0
total_supported_images = 0
total_ambiguous = 0

scan_start_time = time.time()


log("\n" + "=" * 80)
log("STAGE 2 — SCANNING AND SAMPLING")
log("=" * 80)


for dataset_index, plan in enumerate(
    scan_plan,
    start=1,
):
    record = plan["record"]
    roots = plan["roots"]
    dataset = record["dataset"]

    dataset_expected = (
        dataset_expected_counts.get(
            dataset,
            0,
        )
    )

    dataset_scanned = 0
    dataset_supported = 0
    dataset_ambiguous = 0
    dataset_start_time = time.time()

    log("\n" + "=" * 80)
    log(
        f"DATASET "
        f"{dataset_index}/{len(scan_plan)}: "
        f"{dataset.upper()}"
    )
    log(
        f"Expected images: "
        f"{dataset_expected:,}"
    )
    log(
        f"Roots/parts: {len(roots)}"
    )
    log("=" * 80)

    for root_index, root in enumerate(
        roots,
        start=1,
    ):
        root_expected = (
            root_expected_counts.get(
                (dataset, str(root)),
                0,
            )
        )

        root_scanned = 0
        root_supported = 0
        root_ambiguous = 0

        root_start_time = time.time()
        last_log_time = root_start_time

        log(
            f"\nStarting part/root "
            f"{root_index}/{len(roots)}"
        )
        log(f"Path: {root}")
        log(
            f"Expected images in this part: "
            f"{root_expected:,}"
        )

        for (
            current_directory,
            directory_names,
            filenames,
        ) in os.walk(root):
            directory_names.sort()
            filenames.sort()

            current_directory = Path(
                current_directory
            )

            for filename in filenames:
                suffix = (
                    Path(filename)
                    .suffix
                    .lower()
                )

                if suffix not in VALID_EXT:
                    continue

                filepath = (
                    current_directory
                    / filename
                )

                root_scanned += 1
                dataset_scanned += 1
                total_scanned += 1

                label = path_label(
                    filepath,
                    dataset,
                )

                if label is None:
                    root_ambiguous += 1
                    dataset_ambiguous += 1
                    total_ambiguous += 1

                    if (
                        len(ambiguous_examples)
                        < MAX_AMBIGUOUS_EXAMPLES
                    ):
                        ambiguous_examples.append({
                            "dataset": dataset,
                            "filepath": str(
                                filepath
                            ),
                            "source_root": str(
                                root
                            ),
                            "reason": (
                                "Could not determine "
                                "label safely from "
                                "dataset/folder structure"
                            ),
                        })

                else:
                    root_supported += 1
                    dataset_supported += 1
                    total_supported_images += 1

                    row = {
                        "dataset": dataset,
                        "filepath": str(
                            filepath
                        ),
                        "label": int(label),
                        "label_norm": (
                            "fake"
                            if label == 1
                            else "real"
                        ),
                        "exposure": record[
                            "exposure"
                        ],
                        "evaluation_role": record[
                            "role"
                        ],
                        "source_root": str(
                            root
                        ),
                    }

                    add_to_reservoir(
                        reservoir=reservoir,
                        key=(
                            dataset,
                            int(label),
                        ),
                        row=row,
                        limit=(
                            MAX_PER_CLASS_PER_DATASET
                        ),
                    )

                current_time = time.time()

                file_interval_reached = (
                    root_scanned
                    % LOG_EVERY_FILES
                    == 0
                )

                time_interval_reached = (
                    current_time
                    - last_log_time
                    >= LOG_EVERY_SECONDS
                )

                if (
                    file_interval_reached
                    or time_interval_reached
                ):
                    root_elapsed = (
                        current_time
                        - root_start_time
                    )

                    total_elapsed = (
                        current_time
                        - scan_start_time
                    )

                    root_rate = (
                        root_scanned
                        / root_elapsed
                        if root_elapsed > 0
                        else 0
                    )

                    total_rate = (
                        total_scanned
                        / total_elapsed
                        if total_elapsed > 0
                        else 0
                    )

                    root_percent = (
                        100
                        * root_scanned
                        / root_expected
                        if root_expected > 0
                        else 0
                    )

                    dataset_percent = (
                        100
                        * dataset_scanned
                        / dataset_expected
                        if dataset_expected > 0
                        else 0
                    )

                    overall_percent = (
                        100
                        * total_scanned
                        / grand_total_expected
                        if grand_total_expected > 0
                        else 0
                    )

                    remaining_images = max(
                        grand_total_expected
                        - total_scanned,
                        0,
                    )

                    eta_seconds = (
                        remaining_images
                        / total_rate
                        if total_rate > 0
                        else 0
                    )

                    real_selected = len(
                        reservoir.get(
                            (dataset, 0),
                            [],
                        )
                    )

                    fake_selected = len(
                        reservoir.get(
                            (dataset, 1),
                            [],
                        )
                    )

                    log("\nProgress update")

                    log(
                        f"  Dataset: {dataset} "
                        f"({dataset_index}/"
                        f"{len(scan_plan)})"
                    )

                    log(
                        f"  Part/root: "
                        f"{root_index}/"
                        f"{len(roots)}"
                    )

                    log(
                        f"  Current directory: "
                        f"{current_directory}"
                    )

                    log(
                        f"  This part: "
                        f"{root_scanned:,}/"
                        f"{root_expected:,} "
                        f"({root_percent:.1f}%)"
                    )

                    log(
                        f"  This dataset: "
                        f"{dataset_scanned:,}/"
                        f"{dataset_expected:,} "
                        f"({dataset_percent:.1f}%)"
                    )

                    log(
                        f"  Overall: "
                        f"{total_scanned:,}/"
                        f"{grand_total_expected:,} "
                        f"({overall_percent:.1f}%)"
                    )

                    log(
                        f"  Safely labelled: "
                        f"{dataset_supported:,}"
                    )

                    log(
                        f"  Ambiguous: "
                        f"{dataset_ambiguous:,}"
                    )

                    log(
                        "  Selected reservoir: "
                        f"real={real_selected:,}, "
                        f"fake={fake_selected:,}"
                    )

                    log(
                        f"  Current rate: "
                        f"{root_rate:,.0f} "
                        "files/second"
                    )

                    log(
                        f"  Total elapsed: "
                        f"{format_duration(total_elapsed)}"
                    )

                    log(
                        f"  Estimated remaining: "
                        f"{format_duration(eta_seconds)}"
                    )

                    last_log_time = current_time

        root_elapsed = (
            time.time()
            - root_start_time
        )

        log(
            f"\nCompleted part/root "
            f"{root_index}/{len(roots)}"
        )

        log(
            f"  Scanned: "
            f"{root_scanned:,}/"
            f"{root_expected:,}"
        )

        log(
            f"  Safely labelled: "
            f"{root_supported:,}"
        )

        log(
            f"  Ambiguous: "
            f"{root_ambiguous:,}"
        )

        log(
            f"  Time: "
            f"{format_duration(root_elapsed)}"
        )

    dataset_elapsed = (
        time.time()
        - dataset_start_time
    )

    scan_summary_rows.append({
        "dataset": dataset,
        "roots": len(roots),
        "expected_images": (
            dataset_expected
        ),
        "images_scanned": (
            dataset_scanned
        ),
        "safely_labelled": (
            dataset_supported
        ),
        "ambiguous": (
            dataset_ambiguous
        ),
        "elapsed_seconds": (
            dataset_elapsed
        ),
        "elapsed_formatted": (
            format_duration(
                dataset_elapsed
            )
        ),
    })

    overall_percent = (
        100
        * total_scanned
        / grand_total_expected
        if grand_total_expected > 0
        else 0
    )

    log(
        f"\nCOMPLETED DATASET: "
        f"{dataset.upper()}"
    )

    log(
        f"  Images scanned: "
        f"{dataset_scanned:,}/"
        f"{dataset_expected:,}"
    )

    log(
        f"  Safely labelled: "
        f"{dataset_supported:,}"
    )

    log(
        f"  Ambiguous: "
        f"{dataset_ambiguous:,}"
    )

    log(
        f"  Dataset time: "
        f"{format_duration(dataset_elapsed)}"
    )

    log(
        f"  Overall progress: "
        f"{overall_percent:.1f}%"
    )


total_scan_elapsed = (
    time.time()
    - scan_start_time
)

log("\n" + "=" * 80)
log("ALL DATASET SCANNING COMPLETED")
log("=" * 80)

log(
    f"Total scanned: "
    f"{total_scanned:,}/"
    f"{grand_total_expected:,}"
)

log(
    f"Safely labelled: "
    f"{total_supported_images:,}"
)

log(
    f"Ambiguous: "
    f"{total_ambiguous:,}"
)

log(
    f"Total scanning time: "
    f"{format_duration(total_scan_elapsed)}"
)


# =========================================================
# Convert reservoirs into final manifest
# =========================================================

selected_rows = []

for heap in reservoir.values():
    selected_rows.extend(
        item[2]
        for item in heap
    )


manifest = pd.DataFrame(
    selected_rows
)

assert len(manifest) > 0, (
    "No safely labelled images were found. "
    "Check the dataset paths and folder structures."
)


manifest = manifest.sort_values(
    [
        "dataset",
        "label",
        "filepath",
    ]
).reset_index(drop=True)


manifest["sample_id"] = [
    f"E4_{index:07d}"
    for index in range(len(manifest))
]


# =========================================================
# Save outputs
# =========================================================

scan_summary = pd.DataFrame(
    scan_summary_rows
)

scan_summary.to_csv(
    OUTPUT_DIR
    / "Experiment4_Scan_Summary.csv",
    index=False,
)

manifest.to_csv(
    OUTPUT_DIR
    / "Experiment4_Manifest.csv",
    index=False,
)


ambiguous = pd.DataFrame(
    ambiguous_examples
)

if len(ambiguous):
    ambiguous.to_csv(
        OUTPUT_DIR
        / "Experiment4_Ambiguous_Unlabeled_Examples.csv",
        index=False,
    )


# =========================================================
# Final reports and validation
# =========================================================

log("\nScan summary:")

display(scan_summary)


log("\nSelected Experiment 4 manifest:")

manifest_counts = pd.crosstab(
    [
        manifest["dataset"],
        manifest["exposure"],
        manifest["evaluation_role"],
    ],
    manifest["label_norm"],
    margins=True,
)

display(manifest_counts)


log(
    f"\nTotal image paths scanned: "
    f"{total_scanned:,}"
)

log(
    "Safely labelled while scanning: "
    f"{total_supported_images:,}"
)

log(
    f"Ambiguous while scanning: "
    f"{total_ambiguous:,}"
)

log(
    "Final deterministically selected "
    f"images: {len(manifest):,}"
)


# Verify OpenFake coverage across four Kaggle parts.
openfake_rows = manifest[
    manifest["dataset"]
    == "openfake"
]

openfake_roots_used = (
    openfake_rows[
        "source_root"
    ].nunique()
    if len(openfake_rows)
    else 0
)

log(
    "OpenFake roots represented in "
    f"selected sample: "
    f"{openfake_roots_used}/4"
)

if openfake_roots_used < 4:
    log(
        "WARNING: Not all four OpenFake parts "
        "are represented in the final sample. "
        "Review the scan summary and ambiguous "
        "path examples."
    )


assert len(manifest) <= (
    len(DATASETS)
    * 2
    * MAX_PER_CLASS_PER_DATASET
), (
    "Manifest exceeded the configured "
    "maximum sample size."
)


log(
    "\nPASS: Deterministic manifest "
    "creation completed."
)

EXPERIMENT 4 DATASET SCAN PLAN
1. stylegan: 1 root(s)
   [1] /kaggle/input/datasets/xhlulu/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/train
2. flickr30k: 1 root(s)
   [1] /kaggle/input/datasets/adityajn105/flickr30k/Images/flickr30k_images
3. deepdetect2025: 1 root(s)
   [1] /kaggle/input/datasets/ayushmandatta1/deepdetect-2025/ddata/train
4. openfake: 1 root(s)
   [1] /kaggle/working/experiment4_openfake_images
5. ffpp: 1 root(s)
   [1] /kaggle/input/datasets/gradientvoyager/faceforensics-c23-extracted-faces-100k/dataset_processed_split
6. legacy_phone: 1 root(s)
   [1] /kaggle/input/datasets/kashirhanif/phone-images
7. celebdf: 1 root(s)
   [1] /kaggle/input/datasets/pranabr0y/celebdf-v2image-dataset/Celeb_V2
8. stable_diffusion: 1 root(s)
   [1] /kaggle/input/datasets/mohannadaymansalah/stable-diffusion-dataaaaaaaaa

STAGE 1 — COUNTING IMAGE FILES

[1/8] Counting stylegan...
  Part/root 1/1
  Path: /kaggle/input/datasets/xhlulu/140k-real-and-fake-faces/real_vs_fake/real-vs-f

,dataset,roots,expected_images,images_scanned,safely_labelled,ambiguous,elapsed_seconds,elapsed_formatted
0,stylegan,1,100000,100000,100000,0,213.660257,03:33
1,flickr30k,1,31783,31783,31783,0,64.386762,01:04
2,deepdetect2025,1,90409,90409,90409,0,131.298822,02:11
3,openfake,1,1998,1998,1998,0,0.059273,00:00
4,ffpp,1,182246,182246,156081,26165,260.921763,04:20
5,legacy_phone,1,1060,1060,1060,0,1.458245,00:01
6,celebdf,1,101031,101031,101031,0,218.932143,03:38
7,stable_diffusion,1,9001,9001,9001,0,15.879571,00:15



Selected Experiment 4 manifest:


,,label_norm,fake,real,All
dataset,exposure,evaluation_role,,,
celebdf,previously_evaluated,touched_stress_test,1000,1000,2000
deepdetect2025,frequency_expert_seen,legacy_replay,1000,1000,2000
ffpp,spatial_or_fusion_seen,legacy_replay,1000,1000,2000
flickr30k,frequency_expert_seen,legacy_replay,0,1000,1000
legacy_phone,fusion_seen,legacy_replay,0,1000,1000
openfake,frequency_expert_seen,legacy_replay,1000,998,1998
stable_diffusion,never_used,new_generator_stress,1000,0,1000
stylegan,frequency_expert_seen,legacy_replay,1000,1000,2000
All,,,6000,6998,12998



Total image paths scanned: 517,528
Safely labelled while scanning: 491,363
Ambiguous while scanning: 26,165
Final deterministically selected images: 12,998
OpenFake roots represented in selected sample: 1/4

PASS: Deterministic manifest creation completed.


In [10]:
# CELL 6 — FAST PARALLEL EXACT-DUPLICATE AUDIT

import os
import time
import hashlib
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import pandas as pd
from tqdm.auto import tqdm


# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------

HASH_WORKERS = min(
    8,
    max(2, os.cpu_count() or 2),
)

HASH_CHUNK_SIZE = 4 * 1024 * 1024

HASH_CACHE_PATH = (
    OUTPUT_DIR
    / "Experiment4_SHA256_Cache.csv"
)

AUDITED_MANIFEST_PATH = (
    OUTPUT_DIR
    / "Experiment4_Manifest_Audited.csv"
)


# Use the known path instead of recursively searching
# through every attached dataset.
ANNOTATION_CANDIDATES = [
    Path(
        "/kaggle/input/datasets/kashirhanif/"
        "defakex-dataset/"
        "DefakeX_Experiment0_FINAL_Corrected_Files/"
        "DefakeX_annotations_final.csv"
    ),
    Path(
        "/kaggle/input/datasets/kashirhanif/"
        "defakex-dataset/"
        "DefakeX_annotations_final.csv"
    ),
]


ANNOTATION_PATH = next(
    (
        path
        for path in ANNOTATION_CANDIDATES
        if path.exists()
    ),
    None,
)


print("Hash workers:", HASH_WORKERS)

if ANNOTATION_PATH is not None:
    print(
        "DefakeX annotations:",
        ANNOTATION_PATH,
    )
else:
    print(
        "WARNING: DefakeX annotation CSV was not found "
        "at the known paths."
    )


# ---------------------------------------------------------
# Load known DefakeX SHA-256 hashes
# ---------------------------------------------------------

known_hashes = set()

if ANNOTATION_PATH is not None:
    annotations = pd.read_csv(
        ANNOTATION_PATH,
        low_memory=False,
    )

    for column in (
        "actual_sha256",
        "sha256",
    ):
        if column in annotations.columns:
            values = (
                annotations[column]
                .dropna()
                .astype(str)
                .str.strip()
                .str.lower()
            )

            known_hashes.update(values)

    print(
        f"Loaded {len(known_hashes):,} "
        "known DefakeX hashes."
    )


# ---------------------------------------------------------
# SHA-256 helper
# ---------------------------------------------------------

def sha256_file(filepath):
    hasher = hashlib.sha256()

    try:
        with open(filepath, "rb") as file:
            while True:
                chunk = file.read(
                    HASH_CHUNK_SIZE
                )

                if not chunk:
                    break

                hasher.update(chunk)

        return hasher.hexdigest(), ""

    except Exception as error:
        return (
            "",
            f"{type(error).__name__}: {error}",
        )


# ---------------------------------------------------------
# Load reusable hash cache
# ---------------------------------------------------------

hash_cache = {}

if HASH_CACHE_PATH.exists():
    try:
        cached_hashes = pd.read_csv(
            HASH_CACHE_PATH,
            low_memory=False,
        )

        required_columns = {
            "filepath",
            "sha256",
        }

        if required_columns.issubset(
            cached_hashes.columns
        ):
            valid_cache = cached_hashes[
                cached_hashes["sha256"]
                .fillna("")
                .astype(str)
                .str.len()
                .eq(64)
            ]

            hash_cache = dict(
                zip(
                    valid_cache["filepath"]
                    .astype(str),
                    valid_cache["sha256"]
                    .astype(str),
                )
            )

    except Exception as error:
        print(
            "Could not load existing hash cache:",
            type(error).__name__,
            str(error),
        )


print(
    f"Reusable cached hashes: "
    f"{len(hash_cache):,}"
)


# ---------------------------------------------------------
# Determine which files still need hashing
# ---------------------------------------------------------

filepaths = (
    manifest["filepath"]
    .astype(str)
    .tolist()
)

paths_to_hash = [
    filepath
    for filepath in filepaths
    if filepath not in hash_cache
]


print(
    f"Manifest images: {len(filepaths):,}"
)

print(
    f"Already cached: "
    f"{len(filepaths) - len(paths_to_hash):,}"
)

print(
    f"Need hashing: {len(paths_to_hash):,}"
)


# ---------------------------------------------------------
# Calculate missing hashes in parallel
# ---------------------------------------------------------

hash_errors = []
hash_start_time = time.time()

if paths_to_hash:
    with ThreadPoolExecutor(
        max_workers=HASH_WORKERS
    ) as executor:
        results = executor.map(
            sha256_file,
            paths_to_hash,
        )

        for filepath, result in tqdm(
            zip(paths_to_hash, results),
            total=len(paths_to_hash),
            desc="Calculating SHA-256",
            unit="image",
        ):
            sha256_value, error = result

            if sha256_value:
                hash_cache[
                    filepath
                ] = sha256_value

            if error:
                hash_errors.append({
                    "filepath": filepath,
                    "error": error,
                })


print(
    "Hashing completed in",
    f"{(time.time() - hash_start_time) / 60:.2f}",
    "minutes.",
)


# ---------------------------------------------------------
# Save hash cache for recovery/reuse
# ---------------------------------------------------------

hash_cache_dataframe = pd.DataFrame({
    "filepath": list(
        hash_cache.keys()
    ),
    "sha256": list(
        hash_cache.values()
    ),
})

hash_cache_dataframe.to_csv(
    HASH_CACHE_PATH,
    index=False,
)


if hash_errors:
    pd.DataFrame(
        hash_errors
    ).to_csv(
        OUTPUT_DIR
        / "Experiment4_Hash_Errors.csv",
        index=False,
    )


# ---------------------------------------------------------
# Attach hashes to manifest
# ---------------------------------------------------------

manifest["sha256"] = (
    manifest["filepath"]
    .astype(str)
    .map(hash_cache)
    .fillna("")
)


invalid_hash_rows = (
    manifest["sha256"]
    .astype(str)
    .str.len()
    .ne(64)
)

print(
    "Hash failures:",
    int(invalid_hash_rows.sum()),
)


# Files that cannot be hashed cannot safely enter evaluation.
if invalid_hash_rows.any():
    manifest = manifest[
        ~invalid_hash_rows
    ].copy()


manifest[
    "exact_duplicate_of_defakex"
] = (
    manifest["sha256"]
    .str.lower()
    .isin(known_hashes)
)


manifest[
    "duplicate_within_experiment4"
] = manifest.duplicated(
    "sha256",
    keep=False,
)


# ---------------------------------------------------------
# Audit report before filtering
# ---------------------------------------------------------

audit = (
    manifest
    .groupby("dataset")
    .agg(
        images=(
            "sample_id",
            "size",
        ),
        duplicates_of_defakex=(
            "exact_duplicate_of_defakex",
            "sum",
        ),
        within_e4_duplicate_rows=(
            "duplicate_within_experiment4",
            "sum",
        ),
    )
    .reset_index()
)


display(audit)


# ---------------------------------------------------------
# Remove exact byte-level duplicates
# ---------------------------------------------------------

before_count = len(manifest)

defakex_duplicates = int(
    manifest[
        "exact_duplicate_of_defakex"
    ].sum()
)


manifest = (
    manifest[
        ~manifest[
            "exact_duplicate_of_defakex"
        ]
    ]
    .drop_duplicates(
        "sha256",
        keep="first",
    )
    .reset_index(drop=True)
)


removed_count = (
    before_count
    - len(manifest)
)


manifest.to_csv(
    AUDITED_MANIFEST_PATH,
    index=False,
)


print(
    f"Exact DefakeX duplicates removed: "
    f"{defakex_duplicates:,}"
)

print(
    f"Total duplicate/error rows removed: "
    f"{removed_count:,}"
)

print(
    f"Proceeding with "
    f"{len(manifest):,} unique images."
)

print(
    "Saved:",
    AUDITED_MANIFEST_PATH,
)

Hash workers: 4
DefakeX annotations: /kaggle/input/datasets/kashirhanif/defakex-dataset/DefakeX_Experiment0_FINAL_Corrected_Files/DefakeX_annotations_final.csv
Loaded 5,917 known DefakeX hashes.
Reusable cached hashes: 0
Manifest images: 12,998
Already cached: 0
Need hashing: 12,998


Calculating SHA-256:   0%|          | 0/12998 [00:00<?, ?image/s]

Hashing completed in 1.28 minutes.
Hash failures: 0


,dataset,images,duplicates_of_defakex,within_e4_duplicate_rows
0,celebdf,2000,0,0
1,deepdetect2025,2000,0,39
2,ffpp,2000,0,0
3,flickr30k,1000,0,0
4,legacy_phone,1000,13,12
5,openfake,1998,0,2
6,stable_diffusion,1000,0,0
7,stylegan,2000,0,39


Exact DefakeX duplicates removed: 13
Total duplicate/error rows removed: 59
Proceeding with 12,939 unique images.
Saved: /kaggle/working/defakex_experiment_4/Experiment4_Manifest_Audited.csv


In [11]:
# CELL 7 — BUILD AND STRICTLY LOAD BOTH EFFICIENTNET-B3 EXPERTS
def unwrap_checkpoint(obj):
    if isinstance(obj,dict):
        for k in ('state_dict','model_state_dict','model','net'):
            if k in obj and isinstance(obj[k],dict): return obj[k]
    return obj

def normalize_keys(sd):
    out={}
    for k,v in sd.items():
        nk=k
        for prefix in ('module.','model.','backbone.'):
            if nk.startswith(prefix): nk=nk[len(prefix):]
        out[nk]=v
    return out

class EfficientNetB3Binary(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=models.efficientnet_b3(weights=None)
        self.net.classifier[1]=nn.Linear(self.net.classifier[1].in_features,1)
    def forward(self,x,return_embedding=False):
        z=self.net.features(x); z=self.net.avgpool(z); z=torch.flatten(z,1)
        y=self.net.classifier(z).squeeze(1)
        return (y,z) if return_embedding else y

def load_expert(path):
    model=EfficientNetB3Binary()
    obj=torch.load(path,map_location='cpu',weights_only=False)
    sd=normalize_keys(unwrap_checkpoint(obj))
    try: model.load_state_dict(sd,strict=True)
    except RuntimeError:
        # Some checkpoints store torchvision keys without the wrapper's `net.` prefix.
        sd2={('net.'+k if not k.startswith('net.') else k):v for k,v in sd.items()}
        model.load_state_dict(sd2,strict=True)
    model.eval().to(DEVICE)
    return model

frequency_model=load_expert(FREQUENCY_CKPT)
spatial_model=load_expert(SPATIAL_CKPT)
fusion_model=joblib.load(FUSION_MODEL)
print('PASS: strict checkpoint loading succeeded; fusion=',type(fusion_model).__name__)

PASS: strict checkpoint loading succeeded; fusion= Pipeline


In [12]:
# CELL 8 — EXACT FREQUENCY AND SPATIAL PREPROCESSING
to_256=transforms.Resize(256,interpolation=transforms.InterpolationMode.BICUBIC)
center_224=transforms.CenterCrop(224)
spatial_tensor=transforms.Compose([to_256,center_224,transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
mtcnn=MTCNN(keep_all=True,device=DEVICE,post_process=False)

def frequency_tensor(img):
    img=center_224(to_256(img.convert('RGB')))
    x=np.asarray(img,dtype=np.float32)/255.0
    chans=[]
    for c in range(3):
        mag=np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(x[:,:,c]))))
        chans.append(np.clip(mag,0,14)/14.0)
    return torch.from_numpy(np.stack(chans).astype(np.float32))

def largest_face_crop(img,margin=0.25):
    boxes,probs=mtcnn.detect(img)
    if boxes is None or len(boxes)==0:return None,0.0,0
    areas=(boxes[:,2]-boxes[:,0])*(boxes[:,3]-boxes[:,1]);i=int(np.argmax(areas));b=boxes[i]
    cx=(b[0]+b[2])/2;cy=(b[1]+b[3])/2;side=max(b[2]-b[0],b[3]-b[1])*(1+2*margin)
    l=max(0,cx-side/2);t=max(0,cy-side/2);r=min(img.width,cx+side/2);bot=min(img.height,cy+side/2)
    return img.crop((l,t,r,bot)),float(probs[i] if probs is not None else 0),len(boxes)

In [13]:
# CELL 9 — EXTRACT FROZEN EMBEDDINGS
records=[]
with torch.inference_mode():
    for i,row in manifest.iterrows():
        try:
            img=Image.open(row.filepath).convert('RGB')
            fx=frequency_tensor(img).unsqueeze(0).to(DEVICE)
            _,fe=frequency_model(fx,return_embedding=True)
            crop,conf,count=largest_face_crop(img)
            found=crop is not None
            if found:
                sx=spatial_tensor(crop).unsqueeze(0).to(DEVICE)
                _,se=spatial_model(sx,return_embedding=True)
                se=se.cpu().numpy()[0]
            else: se=np.zeros(1536,dtype=np.float32)
            records.append({**row.to_dict(),'face_found':found,'face_confidence':conf,'face_count':count,
                            'frequency_embedding':fe.cpu().numpy()[0],'spatial_embedding':se,'read_error':''})
        except Exception as e:
            records.append({**row.to_dict(),'face_found':False,'face_confidence':0.0,'face_count':0,
                            'frequency_embedding':None,'spatial_embedding':None,'read_error':f'{type(e).__name__}: {e}'})
        if (i+1)%100==0: print(f'{i+1}/{len(manifest)}')

features=pd.DataFrame(records)
errors=features[features.read_error.ne('')].copy(); valid=features[features.read_error.eq('')].copy()
errors.drop(columns=['frequency_embedding','spatial_embedding']).to_csv(OUTPUT_DIR/'Experiment4_Read_Errors.csv',index=False)
assert len(valid)>0
X=np.stack([np.concatenate([a,b]) for a,b in zip(valid.frequency_embedding,valid.spatial_embedding)]).astype(np.float32)
assert X.shape[1]==3072
valid['fake_probability']=fusion_model.predict_proba(X)[:,1]
valid['prediction']=(valid.fake_probability>=THRESHOLD).astype(int)
valid['correct']=(valid.prediction==valid.label).astype(int)
valid.drop(columns=['frequency_embedding','spatial_embedding']).to_csv(OUTPUT_DIR/'Experiment4_Predictions.csv',index=False)
np.savez_compressed(OUTPUT_DIR/'Experiment4_External_Features.npz',X=X,sample_id=valid.sample_id.to_numpy())
print('Valid:',len(valid),'Errors:',len(errors),'No face:',int((~valid.face_found).sum()))

100/12939
200/12939
300/12939
400/12939
500/12939
600/12939
700/12939
800/12939
900/12939
1000/12939
1100/12939
1200/12939
1300/12939
1400/12939
1500/12939
1600/12939
1700/12939
1800/12939
1900/12939
2000/12939
2100/12939
2200/12939
2300/12939
2400/12939
2500/12939
2600/12939
2700/12939
2800/12939
2900/12939
3000/12939
3100/12939
3200/12939
3300/12939
3400/12939
3500/12939
3600/12939
3700/12939
3800/12939
3900/12939
4000/12939
4100/12939
4200/12939
4300/12939
4400/12939
4500/12939
4600/12939
4700/12939
4800/12939
4900/12939
5000/12939
5100/12939
5200/12939
5300/12939
5400/12939
5500/12939
5600/12939
5700/12939
5800/12939
5900/12939
6000/12939
6100/12939
6200/12939
6300/12939
6400/12939
6500/12939
6600/12939
6700/12939
6800/12939
6900/12939
7000/12939
7100/12939
7200/12939
7300/12939
7400/12939
7500/12939
7600/12939
7700/12939
7800/12939
7900/12939
8000/12939
8100/12939
8200/12939
8300/12939
8400/12939
8500/12939
8600/12939
8700/12939
8800/12939
8900/12939
9000/12939
9100/12939
9200/129

In [14]:
# CELL 10 — EXPOSURE-AWARE METRICS (NO INVALID CLAIMS)
def metric_row(g):
    y=g.label.astype(int).to_numpy();p=g.fake_probability.to_numpy();pred=(p>=THRESHOLD).astype(int)
    out={'n':len(g),'accuracy':accuracy_score(y,pred),'fake_recall':float(pred[y==1].mean()) if np.any(y==1) else np.nan,
         'real_recall':float((pred[y==0]==0).mean()) if np.any(y==0) else np.nan,
         'roc_auc':roc_auc_score(y,p) if len(np.unique(y))==2 else np.nan,
         'balanced_accuracy':balanced_accuracy_score(y,pred) if len(np.unique(y))==2 else np.nan,
         'macro_f1':f1_score(y,pred,average='macro') if len(np.unique(y))==2 else np.nan,
         'mean_fake_probability':p.mean(),'face_found_rate':g.face_found.mean()}
    if len(np.unique(y))==2:
        tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel();out.update(tn=int(tn),fp=int(fp),fn=int(fn),tp=int(tp))
    return pd.Series(out)

dataset_metrics=valid.groupby(['dataset','exposure','evaluation_role'],dropna=False).apply(metric_row).reset_index()
display(dataset_metrics)
dataset_metrics.to_csv(OUTPUT_DIR/'Experiment4_Dataset_Metrics.csv',index=False)

role_metrics=valid.groupby(['exposure','evaluation_role'],dropna=False).apply(metric_row).reset_index()
display(role_metrics)
role_metrics.to_csv(OUTPUT_DIR/'Experiment4_Exposure_Tier_Metrics.csv',index=False)

print('INTERPRETATION GATE:')
print('- frequency_expert_seen / spatial_or_fusion_seen / fusion_seen = replay only')
print('- previously_evaluated = touched stress test, not pristine external validation')
print('- never_used Stable Diffusion = new-generator stress test, not a balanced production benchmark by itself')

/tmp/ipykernel_48/3102631501.py:14: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  dataset_metrics=valid.groupby(['dataset','exposure','evaluation_role'],dropna=False).apply(metric_row).reset_index()


,dataset,exposure,evaluation_role,level_3,0
0,celebdf,previously_evaluated,touched_stress_test,n,2000.000000
1,celebdf,previously_evaluated,touched_stress_test,accuracy,0.510500
2,celebdf,previously_evaluated,touched_stress_test,fake_recall,0.364000
3,celebdf,previously_evaluated,touched_stress_test,real_recall,0.657000
4,celebdf,previously_evaluated,touched_stress_test,roc_auc,0.515525
...,...,...,...,...,...
87,stylegan,frequency_expert_seen,legacy_replay,face_found_rate,1.000000
88,stylegan,frequency_expert_seen,legacy_replay,tn,764.000000
89,stylegan,frequency_expert_seen,legacy_replay,fp,219.000000
90,stylegan,frequency_expert_seen,legacy_replay,fn,323.000000


/tmp/ipykernel_48/3102631501.py:18: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  role_metrics=valid.groupby(['exposure','evaluation_role'],dropna=False).apply(metric_row).reset_index()


,exposure,evaluation_role,level_2,0
0,frequency_expert_seen,legacy_replay,n,6958.000000
1,frequency_expert_seen,legacy_replay,accuracy,0.678643
2,frequency_expert_seen,legacy_replay,fake_recall,0.477326
3,frequency_expert_seen,legacy_replay,real_recall,0.829189
4,frequency_expert_seen,legacy_replay,roc_auc,0.717204
5,frequency_expert_seen,legacy_replay,balanced_accuracy,0.653257
6,frequency_expert_seen,legacy_replay,macro_f1,0.653335
7,frequency_expert_seen,legacy_replay,mean_fake_probability,0.384990
8,frequency_expert_seen,legacy_replay,face_found_rate,0.893935
9,frequency_expert_seen,legacy_replay,tn,3301.000000


INTERPRETATION GATE:
- frequency_expert_seen / spatial_or_fusion_seen / fusion_seen = replay only
- previously_evaluated = touched stress test, not pristine external validation
- never_used Stable Diffusion = new-generator stress test, not a balanced production benchmark by itself


In [15]:
# CELL 11 — STABLE DIFFUSION + HELD-OUT DEFAKEX-VALIDATION REAL COMPOSITE (OPTIONAL)
# This creates a development cross-generator test only. It does NOT touch the locked 898 benchmark.
# It requires the Experiment 2 cached features/metadata and uses only `split == validation` real rows.
cache=list(INPUT_ROOT.rglob('Experiment2_Frozen_Features.npz'))
meta=list(INPUT_ROOT.rglob('Experiment2_Frozen_Feature_Metadata.csv'))
composite_metrics=pd.DataFrame()
if STABLE_DIFFUSION_CONFIRMED_NEVER_USED and cache and meta and (valid.dataset=='stable_diffusion').any():
    z=np.load(cache[0]);m=pd.read_csv(meta[0],dtype={'image_id':str})
    # Resolve cache keys by semantic names.
    fk=next(k for k in z.files if 'frequency' in k.lower() and 'embedding' in k.lower())
    sk=next(k for k in z.files if 'spatial' in k.lower() and 'embedding' in k.lower())
    real_mask=(m['split'].eq('validation') & m['label_norm'].eq('real')).to_numpy()
    X_real=np.concatenate([z[fk][real_mask],z[sk][real_mask]],axis=1).astype(np.float32)
    p_real=fusion_model.predict_proba(X_real)[:,1]
    sd=valid[valid.dataset.eq('stable_diffusion')]
    y=np.r_[np.zeros(len(p_real),dtype=int),np.ones(len(sd),dtype=int)]
    p=np.r_[p_real,sd.fake_probability.to_numpy()];pred=(p>=THRESHOLD).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    composite_metrics=pd.DataFrame([{'evaluation':'DefakeX_validation_real + never_used_StableDiffusion_fake',
       'n_real':len(p_real),'n_fake':len(sd),'accuracy':accuracy_score(y,pred),'balanced_accuracy':balanced_accuracy_score(y,pred),
       'macro_f1':f1_score(y,pred,average='macro'),'roc_auc':roc_auc_score(y,p),'real_recall':tn/(tn+fp),
       'fake_recall':tp/(tp+fn),'tn':tn,'fp':fp,'fn':fn,'tp':tp}])
    display(composite_metrics)
    composite_metrics.to_csv(OUTPUT_DIR/'Experiment4_New_Generator_Composite_Metrics.csv',index=False)
else:
    print('Composite skipped: attach Exp2 cache + Stable Diffusion, and confirm it was never used.')

,evaluation,n_real,n_fake,accuracy,balanced_accuracy,macro_f1,roc_auc,real_recall,fake_recall,tn,fp,fn,tp
0,DefakeX_validation_real + never_used_StableDif...,449,1000,0.681159,0.758569,0.678845,0.899323,0.962138,0.555,432,17,445,555


In [16]:
# CELL 12 — FAILURE SLICES AND DECISION REPORT
slice_rows=[]
for dataset,g in valid.groupby('dataset'):
    for face,g2 in g.groupby('face_found'):
        slice_rows.append({'dataset':dataset,'face_found':bool(face),'n':len(g2),'accuracy':g2.correct.mean(),
                           'mean_fake_probability':g2.fake_probability.mean()})
slices=pd.DataFrame(slice_rows);slices.to_csv(OUTPUT_DIR/'Experiment4_Face_Slices.csv',index=False);display(slices)

summary={
 'experiment':'DefakeX Experiment 4 — exposure-aware legacy replay and unseen-generator stress test',
 'threshold':THRESHOLD,'selected_fusion':'frequency_plus_spatial_embeddings','locked_898_used':False,
 'experts_retrained':False,'fusion_retrained':False,'selected_images':int(len(manifest)),'valid_predictions':int(len(valid)),
 'read_errors':int(len(errors)),'datasets':dataset_metrics.to_dict('records'),
 'interpretation':{
   'legacy_replay':'Compatibility/catastrophic-regression evidence only; never claim unseen generalization.',
   'touched_stress_test':'Cross-domain stress evidence, but previously observed during research.',
   'new_generator_stress':'Unseen-generator evidence only when never used in any prior expert/fusion development.',
   'next_step':'Use results to design a train-only augmentation ablation; keep locked 898 untouched until final selection.'}}
with open(OUTPUT_DIR/'Experiment4_Summary.json','w') as f:json.dump(summary,f,indent=2,default=float)
print(json.dumps(summary,indent=2,default=float))

,dataset,face_found,n,accuracy,mean_fake_probability
0,celebdf,False,1,0.000000,0.938800
1,celebdf,True,1999,0.510755,0.467536
2,deepdetect2025,False,1,1.000000,0.374817
3,deepdetect2025,True,1999,0.707354,0.541132
4,ffpp,False,1,0.000000,0.852925
5,ffpp,True,1999,0.421711,0.426064
6,flickr30k,False,328,0.993902,0.083582
7,flickr30k,True,672,0.953869,0.117224
8,legacy_phone,False,455,0.945055,0.186286
9,legacy_phone,True,526,0.916350,0.192362


{
  "experiment": "DefakeX Experiment 4 \u2014 exposure-aware legacy replay and unseen-generator stress test",
  "threshold": 0.665,
  "selected_fusion": "frequency_plus_spatial_embeddings",
  "locked_898_used": false,
  "experts_retrained": false,
  "fusion_retrained": false,
  "selected_images": 12939,
  "valid_predictions": 12939,
  "read_errors": 0,
  "datasets": [
    {
      "dataset": "celebdf",
      "exposure": "previously_evaluated",
      "evaluation_role": "touched_stress_test",
      "level_3": "n",
      "0": 2000.0
    },
    {
      "dataset": "celebdf",
      "exposure": "previously_evaluated",
      "evaluation_role": "touched_stress_test",
      "level_3": "accuracy",
      "0": 0.5105
    },
    {
      "dataset": "celebdf",
      "exposure": "previously_evaluated",
      "evaluation_role": "touched_stress_test",
      "level_3": "fake_recall",
      "0": 0.364
    },
    {
      "dataset": "celebdf",
      "exposure": "previously_evaluated",
      "evaluation_role"

In [17]:
# CELL 13 — PACKAGE RESULTS
(OUTPUT_DIR/'README.txt').write_text(
 'Experiment 4 is exposure-aware. Legacy datasets previously used by experts are replay-only and cannot support unseen-generalization claims. '
 'The locked 898-image DefakeX benchmark was not loaded, tuned on, or evaluated.\n')
archive=shutil.make_archive('/kaggle/working/DefakeX_Experiment4_Exposure_Aware_Replay','zip',OUTPUT_DIR)
print('ZIP:',archive)
try:
    from IPython.display import FileLink,display
    display(FileLink(archive))
except Exception: pass

ZIP: /kaggle/working/DefakeX_Experiment4_Exposure_Aware_Replay.zip


/kaggle/working/DefakeX_Experiment4_Exposure_Aware_Replay.zip